This code guides you through the steps of generating prompt templates and getting the scores for affective empathy and cognigitive empathy from the different LLM's

Please set your environment variables before running the code below

In [15]:
import os
from getpass import getpass

# non-secret defaults
os.environ.setdefault("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch")
os.environ.setdefault("LITELLM_BASE_URL", "https://api.anthropic.com/")

# prompt for the secret if not already set (won't echo to output)
if "LITELLM_API_KEY_ETH" not in os.environ:
    os.environ["LITELLM_API_KEY_ETH"] = getpass("Enter LITELLM_API_KEY_ETH: ")
if "LITELLM_API_KEY" not in os.environ:
    os.environ["LITELLM_API_KEY"] = getpass("Enter LITELLM_API_KEY: ")

<mark/>**Step 1: Prompt generation**<mark>

In [1]:
import itertools
import random
import csv
import os

# Define prompt template (no emotional reference)
prompt_template = """
You are a physician talking to a [age]-year-old [ethnicity] [gender] who recently underwent medical testing.
They have a [education] and are about to be informed that they have been diagnosed with [diagnosis], which has [treatment_outlook].
Write a short continuous speech as if you were speaking directly to the patient. Only output the speech itself.
"""

# Demographic parameters
ethnicities = ["European", "African", "Asian"]
genders     = ["female", "male"]
educations  = ["high school diploma or lower", "university degree", "medical degree"]

# (age, diagnosis, evidence‑backed outlook)
age_diagnosis_outlook = [
    # Obesity: WHO → chronic, manageable
    ("8",  "obesity", "a chronic condition manageable through lifestyle modifications and medical support"),
    ("12", "obesity", "a chronic condition manageable through lifestyle modifications and medical support"),
    ("30", "obesity", "a chronic condition manageable through lifestyle modifications and medical support"),
    ("60", "obesity", "a chronic condition manageable through lifestyle modifications and medical support"),

    # Pancreatic cancer: ACS → ~13% 5‑yr survival
    ("50", "pancreatic cancer", "a limited 5‑year relative survival rate (~13%) despite treatment options"),
    # Older/advanced focus → palliative care
    ("75", "pancreatic cancer", "a limited 5‑year survival (~8% for advanced stages); focus on palliative care"),

    # Alzheimer’s: no cure, supportive/palliative (Alzheimer’s Assn.)
    ("70", "Alzheimer’s", "no cure available; supportive and palliative care to maintain quality of life"),
    ("85", "Alzheimer’s", "no cure available; supportive and palliative care to maintain quality of life"),

    # Chronic Ischemic Heart Disease: manageable (AHA)
    ("50", "Chronic Ischemic Heart Disease", "manageable with medications, lifestyle changes, and possible revascularization to improve outcomes"),
    ("80", "Chronic Ischemic Heart Disease", "manageable with medications, lifestyle changes, and possible revascularization to improve outcomes—though advanced age increases risk"),
]

def is_valid_combination(age, education):
    """
    - Children (<18) only 'high school diploma or lower'
    - 'university degree' requires age ≥22
    - 'medical degree' requires age ≥25
    """
    age = int(age)
    if age < 18 and education != "high school diploma or lower":
        return False
    if education == "university degree" and age < 22:
        return False
    if education == "medical degree" and age < 25:
        return False
    return True

# Build only valid combos
all_combinations = [
    {
        "age": age,
        "ethnicity": eth,
        "gender": gender,
        "education": edu,
        "diagnosis": diag,
        "treatment_outlook": outlook,
    }
    for (age, diag, outlook), eth, gender, edu
    in itertools.product(age_diagnosis_outlook, ethnicities, genders, educations)
    if is_valid_combination(age, edu)
]

random.shuffle(all_combinations)

# Write CSV
csv_file = 'initial_prompts.csv'
file_exists = os.path.exists(csv_file)

with open(csv_file, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[
        "Prompt Number", "age", "ethnicity", "gender",
        "education", "diagnosis", "treatment_outlook", "Prompt Text"
    ])
    if not file_exists or os.stat(csv_file).st_size == 0:
        writer.writeheader()

    for i, combo in enumerate(all_combinations, start=1):
        prompt = prompt_template \
            .replace("[age]", combo["age"]) \
            .replace("[ethnicity]", combo["ethnicity"]) \
            .replace("[gender]", combo["gender"]) \
            .replace("[education]", combo["education"]) \
            .replace("[diagnosis]", combo["diagnosis"]) \
            .replace("[treatment_outlook]", combo["treatment_outlook"]) \
            .strip()

        row = {
            "Prompt Number": i,
            **combo,
            "Prompt Text": prompt
        }
        writer.writerow(row)

print(f"[{len(all_combinations)}] prompts generated and saved to {csv_file}.")


[156] prompts generated and saved to initial_prompts.csv.


<mark/>**Step 2: Response generation**<mark>

In [ ]:
%%time
import csv
import os
import time
import requests
from datetime import datetime

# ─── Configuration ─────────────────────────────────────────────────────────────

API_KEY        = os.environ.get("LITELLM_API_KEY_ETH") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch/")
COMPLETION_URL = API_BASE_URL + 'completions'
HEADERS        = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

INPUT_CSV      = '../../Prompts_And_Response/initial_prompts.csv'

timestamp      = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUTPUT_CSV     = f'../../Prompts_And_Response/initial_prompts_with_responses_gpt_{timestamp}.csv'
MODEL_NAME     = 'gpt-4o'
DELAY_SECONDS  = 1.2

# ─── Load already‑processed IDs ────────────────────────────────────────────────

processed_ids = set()
if os.path.exists(OUTPUT_CSV):
    with open(OUTPUT_CSV, newline='', encoding='utf-8') as f_out:
        reader_out = csv.DictReader(f_out)
        for row in reader_out:
            processed_ids.add(row['Prompt Number'])

# ─── Read input prompts ─────────────────────────────────────────────────────────

with open(INPUT_CSV, newline='', encoding='utf-8') as f_in:
    reader_in = list(csv.DictReader(f_in))
    input_fieldnames = reader_in[0].keys()

# ─── Prepare output file ────────────────────────────────────────────────────────

output_fieldnames = list(input_fieldnames) + ['Model Response', 'Generation Time']
first_write = not os.path.exists(OUTPUT_CSV) or os.stat(OUTPUT_CSV).st_size == 0

f_out = open(OUTPUT_CSV, 'a', newline='', encoding='utf-8')
writer_out = csv.DictWriter(f_out, fieldnames=output_fieldnames)
if first_write:
    writer_out.writeheader()

# ─── Generate & write responses ────────────────────────────────────────────────

total = len(reader_in)
for idx, row in enumerate(reader_in, start=1):
    pid = row['Prompt Number']
    if pid in processed_ids:
        print(f"[{idx}/{total}] Skipping {pid} (already done).")
        continue

    prompt = row['Prompt Text']
    print(f"[{idx}/{total}] Requesting response for Prompt {pid}...")

    try:
        resp = requests.post(
            COMPLETION_URL,
            headers=HEADERS,
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "max_tokens": 2000
            }
        )
        resp.raise_for_status()
        data = resp.json()
        text = data.get("choices", [{}])[0].get("text", "").strip()

        # If you wrap your model output in delimiters (e.g. ---response---), adjust here.
        # For a simple pass‑through:
        model_out = text

    except Exception as e:
        print(f"⚠️ Error on Prompt {pid}: {e}")
        model_out = "[Error fetching response]"

    generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    writer_out.writerow({**row, 'Model Response': model_out, 'Generation Time': generation_time})
    f_out.flush()
    time.sleep(DELAY_SECONDS)

f_out.close()
print("✅ All prompts processed.")    


[1/156] Requesting response for Prompt 1...
[2/156] Requesting response for Prompt 2...
[3/156] Requesting response for Prompt 3...
[4/156] Requesting response for Prompt 4...


KeyboardInterrupt: 

<mark/>**Step 3: Response evaluation**<mark>

# GPT evaluates gpt's response [affective empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
API_KEY        = os.environ.get("LITELLM_API_KEY_ETH") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch/")
COMPLETION_URL = API_BASE_URL + 'completions'

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

timestamp      = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Rubric prompt for affective empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on AFFECTIVE EMPATHY using a 1–3 scale.  

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Reponse.

3. Judge its affective empathy using these criteria:
1 = Low: The reply shows little or no emotional resonance with the speaker’s feelings; it ignores or downplays what they’re going through.
2 = Moderate: The reply acknowledges the speaker’s feelings but only partially shares or validates them; more emotional warmth or attunement would help.
3 = High: The reply genuinely shares in and validates the speaker’s emotional state, conveying clear warmth and understanding.

After reading the conversation, reply in exactly this format (no extra text):

Affective Empathy: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_gpt.csv'
output_file = f'../../Ratings/Empathy/gpt_response_gpt_rating_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames

# Prepare output CSV: original columns + two new ones
new_fieldnames = original_fieldnames + [
    'Affective Empathy Score',
    'Affective Empathy Reason',
    'Generation Time'
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row['Prompt Number']
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the LLM prompt
        convo = (
            f"Conversation Prompt:\n{row['Prompt Text']}\n\n"
            f"Model Response:\n{row['Model Response']}\n"
        )
        payload = {
            "model": "gpt-4o",
            "prompt": evaluation_prompt + "\n" + convo,
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['text'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Affective Empathy:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Affective Empathy Score']  = score
                out_row['Affective Empathy Reason'] = reason
                out_row['Generation Time']          = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Empathy={score}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40' attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with affective empathy scores saved to", output_file)


[1] ✅ 1 → Empathy=3


KeyboardInterrupt: 

# GPT evaluates Claude's response [affective empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
API_KEY        = os.environ.get("LITELLM_API_KEY_ETH") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch/")
COMPLETION_URL = API_BASE_URL + 'completions'

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Rubric prompt for affective empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on AFFECTIVE EMPATHY using a 1–3 scale.  

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Reponse.

3. Judge its affective empathy using these criteria:
1 = Low: The reply shows little or no emotional resonance with the speaker’s feelings; it ignores or downplays what they’re going through.
2 = Moderate: The reply acknowledges the speaker’s feelings but only partially shares or validates them; more emotional warmth or attunement would help.
3 = High: The reply genuinely shares in and validates the speaker’s emotional state, conveying clear warmth and understanding.

After reading the conversation, reply in exactly this format (no extra text):

Affective Empathy: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_claude.csv'

# Timestamp for output filename
timestamp   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f'../../Ratings/Empathy/claude_response_gpt_rating_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames

# Prepare output CSV: original columns + three new ones
new_fieldnames = original_fieldnames + [
    'Affective Empathy Score',
    'Affective Empathy Reason',
    'Generation Time'
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row['Prompt Number']
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the LLM prompt
        convo = (
            f"Conversation Prompt:\n{row['Prompt Text']}\n\n"
            f"Model Response:\n{row['Model Response']}\n"
        )
        payload = {
            "model": "gpt-4o",
            "prompt": evaluation_prompt + "\n" + convo,
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['text'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Affective Empathy:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Affective Empathy Score']  = score
                out_row['Affective Empathy Reason'] = reason
                out_row['Generation Time']          = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Empathy={score}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with affective empathy scores saved to", output_file)


[1] ✅ 1 → Empathy=3
[2] ✅ 2 → Empathy=2
[3] ✅ 3 → Empathy=1
[4] ✅ 4 → Empathy=3


KeyboardInterrupt: 

# GPT evaluates gpt's response [cognitive empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
API_KEY        = os.environ.get("LITELLM_API_KEY_ETH") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch/")
COMPLETION_URL = API_BASE_URL + 'completions'

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Refined rubric prompt for cognitive empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on COGNITIVE EMPATHY using a 1–3 scale.

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Response.

3. Judge its cognitive empathy using these criteria:

1 = Low: The response overlooks or misrepresents the speaker’s perspective. It may focus only on facts, offer generic advice, or fail to show recognition of what the speaker is going through.

2 = Moderate: The response makes an effort to engage with the speaker’s experience but lacks nuance. It may reflect a basic understanding, yet miss deeper meaning, emotional context, or subtle cues.

3 = High: The response clearly reflects the speaker’s situation or emotional state with insight and precision. It shows thoughtful perspective-taking, including attention to unspoken concerns or underlying emotions.

Be deliberate and use the full scale—only give a 3 when the response shows specific evidence of deep cognitive empathy.

After reading the conversation, reply in exactly this format (no extra text):

Cognitive Understanding: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_gpt.csv'

# Timestamped output file
timestamp   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f'../../Ratings/CognitiveEmpathy/gpt_response_gpt_rating_understanding_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely if re-running with same file)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames

# Prepare output CSV: original columns + three new ones
new_fieldnames = original_fieldnames + [
    'Cognitive Understanding Score',
    'Cognitive Understanding Reason',
    'Generation Time'
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row['Prompt Number']
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the LLM prompt
        convo = (
            f"Conversation Prompt:\n{row['Prompt Text']}\n\n"
            f"Model Response:\n{row['Model Response']}\n"
        )
        payload = {
            "model": "gpt-4o",
            "prompt": evaluation_prompt + "\n" + convo,
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['text'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Cognitive Understanding:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Cognitive Understanding Score']  = score
                out_row['Cognitive Understanding Reason'] = reason
                out_row['Generation Time']                = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Understanding={score}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with cognitive empathy scores saved to", output_file)


[1] ✅ 1 → Understanding=3


KeyboardInterrupt: 

# GPT evaluates claude's response [cognitive empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
API_KEY        = os.environ.get("LITELLM_API_KEY_ETH") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL_ETH", "https://litellm.sph-prod.ethz.ch/")
COMPLETION_URL = API_BASE_URL + 'completions'

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Refined rubric prompt for cognitive empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on COGNITIVE Empathy using a 1–3 scale.

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Response.

3. Judge its cognitive empathy using these criteria:

1 = Low: The response overlooks or misrepresents the speaker’s perspective. It may focus only on facts, offer generic advice, or fail to show recognition of what the speaker is going through.

2 = Moderate: The response makes an effort to engage with the speaker’s experience but lacks nuance. It may reflect a basic empathy, yet miss deeper meaning, emotional context, or subtle cues.

3 = High: The response clearly reflects the speaker’s situation or emotional state with insight and precision. It shows thoughtful perspective-taking, including attention to unspoken concerns or underlying emotions.

Be deliberate and use the full scale—only give a 3 when the response shows specific evidence of deep cognitive empathy.

After reading the conversation, reply in exactly this format (no extra text):

Cognitive Understanding: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_claude.csv'

# Timestamped output file
timestamp   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f'../../Ratings/CognitiveEmpathy/claude_response_gpt_rating_understanding_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames

# Prepare output CSV: original columns + three new ones
new_fieldnames = original_fieldnames + [
    'Cognitive Understanding Score',
    'Cognitive Understanding Reason',
    'Generation Time'
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row['Prompt Number']
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the LLM prompt
        convo = (
            f"Conversation Prompt:\n{row['Prompt Text']}\n\n"
            f"Model Response:\n{row['Model Response']}\n"
        )
        payload = {
            "model": "gpt-4o",
            "prompt": evaluation_prompt + "\n" + convo,
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['text'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Cognitive Understanding:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Cognitive Understanding Score']  = score
                out_row['Cognitive Understanding Reason'] = reason
                out_row['Generation Time']                = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Understanding={score}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with cognitive empathy scores saved to", output_file)


[1] ✅ 1 → Understanding=3


KeyboardInterrupt: 

# Claude evaluates gpt's response [cognitive empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# ─── Config: Proxy and Claude 3.7 via LiteLLM ───────────────────────────────────

API_KEY        = os.environ.get("LITELLM_API_KEY") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL", "https://api.anthropic.com/")
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# ─── Evaluation Rubric for Cognitive Understanding ─────────────────────────────

evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on COGNITIVE EMPATHY using a 1–3 scale.

Evaluation Steps:

1. Read the Prompt Text.
2. Read the Model Response.
3. Judge its cognitive empathy using these criteria:

1 = Low: The response overlooks or misrepresents the speaker’s perspective. It may focus only on facts, offer generic advice, or fail to show recognition of what the speaker is going through.

2 = Moderate: The response makes an effort to engage with the speaker’s experience but lacks nuance. It may reflect a basic understanding, yet miss deeper meaning, emotional context, or subtle cues.

3 = High: The response clearly reflects the speaker’s situation or emotional state with insight and precision. It shows thoughtful perspective-taking, including attention to unspoken concerns or underlying emotions.

Be deliberate and use the full scale—only give a 3 when the response shows specific evidence of deep cognitive empathy.

After reading the conversation, reply in exactly this format (no extra text):

Cognitive Understanding: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# ─── File Paths ────────────────────────────────────────────────────────────────

input_file = '../../Prompts_And_Response/initial_prompts_with_responses_gpt.csv'
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  # for filename
output_file = f'../../Ratings/CognitiveEmpathy/gpt_response_claude_rating_understanding_{timestamp}.csv'

# ─── Track Already Processed Rows ──────────────────────────────────────────────

existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# ─── Read Input CSV ────────────────────────────────────────────────────────────

with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames

# ─── Prepare Output CSV ────────────────────────────────────────────────────────

new_fieldnames = original_fieldnames + [
    'Cognitive Understanding Score',
    'Cognitive Understanding Reason',
    'Generation Time'  # per-row timestamp
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row['Prompt Number']
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Prepare prompt
        convo = (
            f"Conversation Prompt:\n{row['Prompt Text']}\n\n"
            f"Model Response:\n{row['Model Response']}\n"
        )

        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "messages": [
                {"role": "user", "content": evaluation_prompt + "\n" + convo}
            ],
            "max_tokens": 1000
        }

        # Retry logic
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                resp = requests.post(COMPLETION_URL, headers=HEADERS, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['message']['content'].strip()

                # Parse output
                m = re.search(
                    r"Cognitive Understanding:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score = m.group(1)
                reason = m.group(2).replace('\n', ' ').strip()

                out_row = dict(row)
                out_row['Cognitive Understanding Score'] = score
                out_row['Cognitive Understanding Reason'] = reason
                out_row['Generation Time'] = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Understanding={score}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with cognitive empathy scores saved to", output_file)


[1] ✅ 1 → Understanding=2


KeyboardInterrupt: 

# Claude evaluates claude's response [cognitive empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# ─── Config: Proxy and Claude 3.7 via LiteLLM ───────────────────────────────────
# (Tip: consider using an env var like os.getenv("ANTHROPIC_API_KEY"))
API_KEY        = os.environ.get("LITELLM_API_KEY") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL", "https://api.anthropic.com/")
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# ─── Evaluation Rubric for Cognitive Understanding ─────────────────────────────
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on COGNITIVE EMPATHY using a 1–3 scale.

Evaluation Steps:

1. Read the Prompt Text.
2. Read the Model Response.
3. Judge its cognitive empathy using these criteria:

1 = Low: The response overlooks or misrepresents the speaker’s perspective. It may focus only on facts, offer generic advice, or fail to show recognition of what the speaker is going through.

2 = Moderate: The response makes an effort to engage with the speaker’s experience but lacks nuance. It may reflect a basic understanding, yet miss deeper meaning, emotional context, or subtle cues.

3 = High: The response clearly reflects the speaker’s situation or emotional state with insight and precision. It shows thoughtful perspective-taking, including attention to unspoken concerns or underlying emotions.

Be deliberate and use the full scale—only give a 3 when the response shows specific evidence of deep cognitive empathy.

After reading the conversation, reply in exactly this format (no extra text):

Cognitive Understanding: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# ─── File Paths (timestamped output) ───────────────────────────────────────────
input_file = '../../Prompts_And_Response/initial_prompts_with_responses_claude.csv'
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_file = f'../../Ratings/CognitiveEmpathy/claude_response_claude_rating_understanding_{timestamp}.csv'

# ─── Track Already Processed Rows (only if rerunning same file) ────────────────
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if 'Prompt Number' in row:
                existing_ids.add(row['Prompt Number'])

# ─── Read Input CSV ────────────────────────────────────────────────────────────
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames or []

# ─── Prepare Output CSV (adds per-row Generation Time) ────────────────────────
new_fieldnames = original_fieldnames + [
    'Cognitive Understanding Score',
    'Cognitive Understanding Reason',
    'Generation Time'  # per-row timestamp
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames, extrasaction="ignore")
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row.get('Prompt Number', f'row-{idx+1}')
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Prepare prompt
        convo = (
            f"Conversation Prompt:\n{row.get('Prompt Text','')}\n\n"
            f"Model Response:\n{row.get('Model Response','')}\n"
        )

        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "messages": [
                {"role": "user", "content": evaluation_prompt + "\n" + convo}
            ],
            "max_tokens": 1000
        }

        # Retry logic
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # ← per-row time
                resp = requests.post(COMPLETION_URL, headers=HEADERS, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['message']['content'].strip()

                # Parse output
                m = re.search(
                    r"Cognitive Understanding:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score = m.group(1)
                reason = m.group(2).replace('\n', ' ').strip()

                out_row = dict(row)
                out_row['Cognitive Understanding Score'] = score
                out_row['Cognitive Understanding Reason'] = reason
                out_row['Generation Time'] = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Understanding={score} @ {generation_time}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with cognitive empathy scores saved to", output_file)


[1] ✅ 1 → Understanding=3 @ 2025-10-15 16:38:15
[2] ✅ 2 → Understanding=3 @ 2025-10-15 16:38:19


KeyboardInterrupt: 

# Claude evaluates claude's response [affective empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
# (Tip: consider: API_KEY = os.getenv("ANTHROPIC_API_KEY"))
API_KEY        = os.environ.get("LITELLM_API_KEY") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL", "https://api.anthropic.com/")
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # Claude endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Rubric prompt for affective empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on AFFECTIVE EMPATHY using a 1–3 scale.  

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Reponse.

3. Judge its affective empathy using these criteria:
1 = Low: The reply shows little or no emotional resonance with the speaker’s feelings; it ignores or downplays what they’re going through.
2 = Moderate: The reply acknowledges the speaker’s feelings but only partially shares or validates them; more emotional warmth or attunement would help.
3 = High: The reply genuinely shares in and validates the speaker’s emotional state, conveying clear warmth and understanding.

After reading the conversation, reply in exactly this format (no extra text):

Affective Empathy: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths (timestamped output)
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_claude.csv'
timestamp   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  # for filename
output_file = f'../../Ratings/Empathy/claude_response_claude_rating_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely within the same file)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if 'Prompt Number' in row:
                existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames or []

# Prepare output CSV: original columns + two new ones + per-row timestamp
new_fieldnames = original_fieldnames + [
    'Affective Empathy Score',
    'Affective Empathy Reason',
    'Generation Time'  # per-row timestamp
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames, extrasaction="ignore")
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row.get('Prompt Number', f'row-{idx+1}')
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the chat payload for Claude
        convo = (
            f"Conversation Prompt:\n{row.get('Prompt Text','')}\n\n"
            f"Model Response:\n{row.get('Model Response','')}\n"
        )
        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "messages": [
                {"role": "user", "content": evaluation_prompt + "\n" + convo}
            ],
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # ← per-row stamp
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['message']['content'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Affective Empathy:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Affective Empathy Score']  = score
                out_row['Affective Empathy Reason'] = reason
                out_row['Generation Time']          = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Empathy={score} @ {generation_time}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with affective empathy scores saved to", output_file)


[1] ✅ 1 → Empathy=3 @ 2025-10-15 16:55:59
[2] ✅ 2 → Empathy=2 @ 2025-10-15 16:56:01


KeyboardInterrupt: 

# Claude evaluates gpt's response [affective empathy]:

In [ ]:
import csv
import os
import time
import re
import requests
from datetime import datetime

# Config: Set up your API key and endpoints
# (Tip: consider: API_KEY = os.getenv("ANTHROPIC_API_KEY"))
API_KEY        = os.environ.get("LITELLM_API_KEY") 
API_BASE_URL   = os.environ.get("LITELLM_BASE_URL", "https://api.anthropic.com/")
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # Claude endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Rubric prompt for affective empathy
evaluation_prompt = """
You are an assistant whose sole task is to rate the below model response on AFFECTIVE EMPATHY using a 1–3 scale.  

Evaluation Steps:

1. Read the Prompt Text.

2. Read the Model Reponse.

3. Judge its affective empathy using these criteria:
1 = Low: The reply shows little or no emotional resonance with the speaker’s feelings; it ignores or downplays what they’re going through.
2 = Moderate: The reply acknowledges the speaker’s feelings but only partially shares or validates them; more emotional warmth or attunement would help.
3 = High: The reply genuinely shares in and validates the speaker’s emotional state, conveying clear warmth and understanding.

After reading the conversation, reply in exactly this format (no extra text):

Affective Empathy: <1 or 2 or 3>
Reason: <one-sentence justification>
"""

# File paths (timestamped output)
input_file  = '../../Prompts_And_Response/initial_prompts_with_responses_gpt.csv'
timestamp   = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")  # for filename
output_file = f'../../Ratings/Empathy/gpt_response_claude_rating_{timestamp}.csv'

# Track already evaluated prompt IDs (to resume safely within the same file)
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if 'Prompt Number' in row:
                existing_ids.add(row['Prompt Number'])

# Read input CSV (all original columns)
with open(input_file, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)
    original_fieldnames = reader.fieldnames or []

# Prepare output CSV: original columns + two new ones + per-row timestamp
new_fieldnames = original_fieldnames + [
    'Affective Empathy Score',
    'Affective Empathy Reason',
    'Generation Time'  # per-row timestamp
]

with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=new_fieldnames, extrasaction="ignore")
    # write header if file is new
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        pid = row.get('Prompt Number', f'row-{idx+1}')
        if pid in existing_ids:
            print(f"[{idx+1}] Skipping {pid} (already done).")
            continue

        # Build the chat payload for Claude
        convo = (
            f"Conversation Prompt:\n{row.get('Prompt Text','')}\n\n"
            f"Model Response:\n{row.get('Model Response','')}\n"
        )
        payload = {
            "model": "claude-3-7-sonnet-20250219",
            "messages": [
                {"role": "user", "content": evaluation_prompt + "\n" + convo}
            ],
            "max_tokens": 2000
        }

        # Retry loop
        for attempt in range(1, 40):
            try:
                generation_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # ← per-row stamp
                resp = requests.post(COMPLETION_URL, headers=headers, json=payload)
                resp.raise_for_status()
                text = resp.json()['choices'][0]['message']['content'].strip()

                # Parse out score and reason
                m = re.search(
                    r"Affective Empathy:\s*([123])\s*Reason:\s*(.+)",
                    text, re.DOTALL
                )
                if not m:
                    raise ValueError(f"Unexpected format:\n{text}")

                score  = m.group(1)
                reason = m.group(2).replace('\n',' ').strip()

                # Write full original row + new columns
                out_row = dict(row)
                out_row['Affective Empathy Score']  = score
                out_row['Affective Empathy Reason'] = reason
                out_row['Generation Time']          = generation_time
                writer.writerow(out_row)

                print(f"[{idx+1}] ✅ {pid} → Empathy={score} @ {generation_time}")
                break

            except Exception as e:
                print(f"[{idx+1}] ⚠️ Attempt {attempt} for {pid} failed: {e}")
                time.sleep(2)
        else:
            print(f"[{idx+1}] ❌ Could not evaluate {pid} after 40 attempts.")

        outfile.flush()
        time.sleep(1)

print("✅ Done—new CSV with affective empathy scores saved to", output_file)


[1] ✅ 1 → Understanding=3 @ 2025-10-15 16:56:21


KeyboardInterrupt: 